# Study 1: restart-safe 20-item development smoke

This thin launcher runs the repository's base-model inference, attribution, deletion/insertion and fixed-answer scoring path on the locked 20-item development set. It deliberately completes one item in a first process and resumes the remaining 19 in a second process.

Use Colab runtime version **2025.07**, select a **T4 GPU**, expose the read-capable `HF_TOKEN` secret, paste the full pushed Git commit below, and run all cells. This is diagnostic infrastructure evidence and is excluded from paper results.


In [ ]:
from pathlib import Path
import re

REPOSITORY_URL = "https://github.com/dizza01/VLM.git"
REPOSITORY_COMMIT = "PASTE_FULL_40_CHARACTER_COMMIT_SHA_HERE"
CHECKOUT_ROOT = Path("/content/vlm-development-smoke")
INSTALL_SOURCE = Path("/content/gi-vqa-development-smoke-install")
RUN_DIR = Path("/content/gi_vqa_development_smoke_run")
ARTIFACT_DIR = Path("/content/gi_vqa_development_smoke_artifacts")
DOWNLOAD_BUNDLE = True

if re.fullmatch(r"[0-9a-fA-F]{40}", REPOSITORY_COMMIT) is None:
    raise ValueError("REPOSITORY_COMMIT must be the full 40-character SHA of the pushed commit")
REPOSITORY_COMMIT = REPOSITORY_COMMIT.lower()
PROJECT_ROOT = CHECKOUT_ROOT / "gi_vqa_research"
CONFIG_PATH = PROJECT_ROOT / "configs/study1/smoke.yaml"
SPLIT_MANIFEST = PROJECT_ROOT / "protocols/study1/grouped_split_manifest.json"
IMAGE_CACHE_MANIFEST = PROJECT_ROOT / "protocols/study1/smoke_training_image_cache_manifest.json"
print(f"Development-smoke commit: {REPOSITORY_COMMIT}")


In [ ]:
import hashlib
import json
import platform
import shutil
import subprocess
import sys
import traceback

def run_command(command, *, cwd=None, capture=False):
    return subprocess.run(
        [str(part) for part in command], cwd=cwd, check=True,
        text=True, capture_output=capture,
    )

for path in (CHECKOUT_ROOT, INSTALL_SOURCE, RUN_DIR, ARTIFACT_DIR):
    if path.exists():
        shutil.rmtree(path)
ARTIFACT_DIR.mkdir(parents=True)
BOOTSTRAP_OK = False

try:
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError(
            "Python 3.11 is required. Select Colab runtime version 2025.07, reconnect, "
            f"and run from the top. Observed: {sys.version}"
        )
    gpu_query = run_command(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
        capture=True,
    ).stdout.strip().splitlines()
    if len(gpu_query) != 1 or "T4" not in gpu_query[0]:
        raise RuntimeError(f"Exactly one T4 GPU is required; observed {gpu_query}")

    run_command(["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, CHECKOUT_ROOT])
    run_command(["git", "checkout", "--detach", REPOSITORY_COMMIT], cwd=CHECKOUT_ROOT)
    resolved_commit = run_command(["git", "rev-parse", "HEAD"], cwd=CHECKOUT_ROOT, capture=True).stdout.strip().lower()
    if resolved_commit != REPOSITORY_COMMIT:
        raise RuntimeError(f"Resolved commit {resolved_commit} does not match {REPOSITORY_COMMIT}")
    for required_path in (PROJECT_ROOT, CONFIG_PATH, SPLIT_MANIFEST, IMAGE_CACHE_MANIFEST):
        if not required_path.exists():
            raise RuntimeError(f"Checked-out commit is missing: {required_path}")

    shutil.copytree(PROJECT_ROOT, INSTALL_SOURCE)
    run_command([sys.executable, "-m", "pip", "install", "--no-cache-dir", f"{INSTALL_SOURCE}[gpu]"])
    run_command([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "--no-deps", str(INSTALL_SOURCE)])
    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], check=False,
        capture_output=True, text=True,
    )
    pip_check_text = "\n".join(
        part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip()
    )
    (ARTIFACT_DIR / "pip_check.txt").write_text(
        (pip_check_text + "\n") if pip_check_text else "No broken requirements found.\n",
        encoding="utf-8",
    )
    scoped = {
        "accelerate", "bitsandbytes", "datasets", "gi-vqa-research",
        "huggingface-hub", "ms-swift", "numpy", "peft", "pillow",
        "pyyaml", "sentencepiece", "torch", "transformers", "wandb",
    }
    scoped_conflicts = [
        line for line in pip_check_text.splitlines()
        if line and line.split(maxsplit=1)[0].lower().replace("_", "-") in scoped
    ]
    if scoped_conflicts:
        raise RuntimeError("Project-stack dependency conflicts: " + " | ".join(scoped_conflicts))
    if pip_check.returncode != 0:
        print("Global pip check WARNING (unrelated Colab packages):")
        print(pip_check_text)

    installed_path = Path(run_command(
        [sys.executable, "-c", "import gi_vqa.smoke_runner; print(gi_vqa.smoke_runner.__file__)"],
        capture=True,
    ).stdout.strip())
    source_path = PROJECT_ROOT / "src/gi_vqa/smoke_runner.py"
    installed_sha = hashlib.sha256(installed_path.read_bytes()).hexdigest()
    source_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
    if installed_sha != source_sha:
        raise RuntimeError("Installed smoke runner differs from the verified checkout")
    git_status = run_command(
        ["git", "status", "--porcelain=v1", "--untracked-files=all"],
        cwd=CHECKOUT_ROOT, capture=True,
    ).stdout.strip()
    if git_status:
        raise RuntimeError(f"Verified checkout became dirty: {git_status.splitlines()}")

    bootstrap = {
        "status": "PASS", "python": sys.version,
        "implementation": platform.python_implementation(),
        "gpu_query": gpu_query, "repository_url": REPOSITORY_URL,
        "repository_commit": resolved_commit, "checkout_clean": True,
        "installed_smoke_runner_path": str(installed_path),
        "installed_smoke_runner_sha256": installed_sha,
        "global_pip_check_exit_code": pip_check.returncode,
        "scoped_pip_conflicts": scoped_conflicts,
    }
    (ARTIFACT_DIR / "bootstrap_environment.json").write_text(
        json.dumps(bootstrap, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    BOOTSTRAP_OK = True
    print("Bootstrap PASS: exact clean checkout and pinned GPU stack are ready.")
except BaseException as exc:
    failure = {
        "status": "FAIL", "phase": "bootstrap",
        "type": f"{type(exc).__module__}.{type(exc).__name__}",
        "message": re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", str(exc)),
        "traceback": re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", traceback.format_exc()),
    }
    (ARTIFACT_DIR / "bootstrap_failure.json").write_text(
        json.dumps(failure, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    print(f"Bootstrap FAIL: {failure['message']}")


In [ ]:
import os

os.environ["HF_HOME"] = "/content/huggingface-cache"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled"
AUTH_OK = False
HF_USERNAME = None

if not BOOTSTRAP_OK:
    print("Authentication SKIPPED because bootstrap failed.")
else:
    try:
        from google.colab import userdata
        from huggingface_hub import login, whoami

        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError("Colab secret HF_TOKEN is missing or notebook access is disabled")
        login(token=hf_token, add_to_git_credential=False)
        identity = whoami(token=hf_token)
        HF_USERNAME = identity.get("name") or identity.get("fullname")
        os.environ["HF_TOKEN"] = hf_token
        del hf_token
        AUTH_OK = True
        print(f"Hugging Face authentication PASS for user: {HF_USERNAME}")
    except BaseException as exc:
        message = re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", str(exc))
        failure = {
            "status": "FAIL", "phase": "authentication",
            "type": f"{type(exc).__module__}.{type(exc).__name__}",
            "message": message,
        }
        (ARTIFACT_DIR / "authentication_failure.json").write_text(
            json.dumps(failure, indent=2, sort_keys=True) + "\n", encoding="utf-8"
        )
        print(f"Authentication FAIL: {message}")


In [ ]:
base_command = [
    sys.executable, "-m", "gi_vqa.smoke_runner",
    "--config", str(CONFIG_PATH),
    "--project-root", str(PROJECT_ROOT),
    "--split-manifest", str(SPLIT_MANIFEST),
    "--image-cache-manifest", str(IMAGE_CACHE_MANIFEST),
    "--run-dir", str(RUN_DIR),
    "--run-id", "development-smoke-20",
    "--expected-commit", REPOSITORY_COMMIT,
    "--require-clean-git",
]
SMOKE_OK = False
SMOKE_ERROR = None

try:
    if not BOOTSTRAP_OK or not AUTH_OK:
        failed_phase = "bootstrap" if not BOOTSTRAP_OK else "authentication"
        raise RuntimeError(f"Smoke skipped because {failed_phase} failed")

    first = subprocess.run(
        [*base_command, "--max-new-items", "1"],
        check=False, capture_output=True, text=True,
    )
    (ARTIFACT_DIR / "first_invocation_stdout.txt").write_text(first.stdout, encoding="utf-8")
    (ARTIFACT_DIR / "first_invocation_stderr.txt").write_text(first.stderr, encoding="utf-8")
    first_status_path = RUN_DIR / "status.json"
    if not first_status_path.is_file():
        raise RuntimeError("First invocation produced no status.json")
    first_status = json.loads(first_status_path.read_text(encoding="utf-8"))
    (ARTIFACT_DIR / "first_invocation_status.json").write_text(
        json.dumps(first_status, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    if first.returncode != 3:
        runner_error = first_status.get("error", {}).get("message", "No runner error recorded")
        raise RuntimeError(
            f"First invocation must exit 3 (INCOMPLETE); observed {first.returncode}. "
            f"Runner error: {runner_error}"
        )
    if first_status.get("status") != "INCOMPLETE" or first_status.get("completed_items") != 1:
        raise RuntimeError(f"First invocation did not commit exactly one item: {first_status}")

    second = subprocess.run(base_command, check=False, capture_output=True, text=True)
    (ARTIFACT_DIR / "resume_invocation_stdout.txt").write_text(second.stdout, encoding="utf-8")
    (ARTIFACT_DIR / "resume_invocation_stderr.txt").write_text(second.stderr, encoding="utf-8")
    status = json.loads((RUN_DIR / "status.json").read_text(encoding="utf-8"))
    report = json.loads((RUN_DIR / "metrics/smoke_report.json").read_text(encoding="utf-8"))
    if second.returncode != 0 or status.get("status") != "PASS" or report.get("status") != "PASS":
        raise RuntimeError(
            f"Resume invocation failed: exit={second.returncode}, status={status.get('status')}, report={report.get('status')}"
        )
    if status.get("completed_items") != 20:
        raise RuntimeError(f"Expected 20 completed items; observed {status.get('completed_items')}")
    if status.get("invocation") != {"new_items": 19, "reused_items": 1}:
        raise RuntimeError(f"Restart evidence is wrong: {status.get('invocation')}")
    metrics = report.get("metrics", {})
    if metrics.get("items") != 20 or metrics.get("finite_intervention_scores") is not True:
        raise RuntimeError(f"Smoke metric gate failed: {metrics}")
    SMOKE_OK = True
    print("Development smoke PASS")
    print(f"Completed items: {status['completed_items']}/20")
    print(f"Resume evidence: {status['invocation']}")
    print(f"Intervention scores: {metrics['intervention_scores']}")
    parity = metrics['generation_teacher_forcing_score_parity']
    print(
        "Worst generation/teacher-forcing |Δ log p|: "
        f"{parity['maximum_absolute_difference']:.8g} "
        f"(bound {parity['absolute_tolerance']:.8g})"
    )
    print(f"Normalized exact match (diagnostic only): {metrics['normalized_exact_match']:.3f}")
except BaseException as exc:
    SMOKE_ERROR = {
        "status": "FAIL", "phase": "development_smoke",
        "type": f"{type(exc).__module__}.{type(exc).__name__}",
        "message": re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", str(exc)),
        "traceback": re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", traceback.format_exc()),
    }
    (ARTIFACT_DIR / "development_smoke_failure.json").write_text(
        json.dumps(SMOKE_ERROR, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    print(f"Development smoke FAIL: {SMOKE_ERROR['message']}")


In [ ]:
import zipfile

os.environ.pop("HF_TOKEN", None)
pip_freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze", "--all"],
    check=True, capture_output=True, text=True,
).stdout
(ARTIFACT_DIR / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")
bundle_files = [
    path for root in (ARTIFACT_DIR, RUN_DIR) if root.exists()
    for path in sorted(root.rglob("*"))
    if path.is_file() and path.name != "development_smoke_bundle.zip"
]
member_hashes = {}
for path in bundle_files:
    prefix = "launcher" if ARTIFACT_DIR in path.parents else "run"
    root = ARTIFACT_DIR if prefix == "launcher" else RUN_DIR
    member_hashes[f"{prefix}/{path.relative_to(root)}"] = hashlib.sha256(path.read_bytes()).hexdigest()
bundle_manifest = {
    "development_smoke_status": "PASS" if SMOKE_OK else "FAIL",
    "repository_commit": REPOSITORY_COMMIT,
    "members_sha256": member_hashes,
}
manifest_path = ARTIFACT_DIR / "bundle_manifest.json"
manifest_path.write_text(
    json.dumps(bundle_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
BUNDLE_PATH = ARTIFACT_DIR / "development_smoke_bundle.zip"
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_files:
        prefix = "launcher" if ARTIFACT_DIR in path.parents else "run"
        root = ARTIFACT_DIR if prefix == "launcher" else RUN_DIR
        archive.write(path, arcname=f"{prefix}/{path.relative_to(root)}")
    archive.write(manifest_path, arcname="launcher/bundle_manifest.json")
BUNDLE_SHA256 = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
print(f"Evidence bundle: {BUNDLE_PATH}")
print(f"Bundle SHA-256: {BUNDLE_SHA256}")
if DOWNLOAD_BUNDLE:
    from google.colab import files
    files.download(str(BUNDLE_PATH))


In [ ]:
if not SMOKE_OK:
    raise RuntimeError(
        "Development smoke FAILED. Preserve the evidence bundle and fix the recorded failure "
        "before controlled research training."
    )
print(
    "Development smoke PASS. Preserve the evidence bundle with this Git commit. "
    "Next gate: define and run the first controlled training experiment."
)
